RUN THIS FIRST BLOCK TO LAUNCH : FILTER WHEEL, LASER AND PICOSCOPE (MAKE SURE THEY ARE ALL DISCONNECTED FROM THEIR SOFTWARES)

In [1]:
import matplotlib.pyplot as plt
import os
import numpy as np
from datetime import datetime
from picosdk.ps6000 import ps6000 as ps
from picosdk.functions import adc2mV, assert_pico_ok
from ctypes import c_int16, c_int32, byref, c_char_p,c_float

#chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://www.picotech.com/download/manuals/picoscope-6000-series-programmers-guide.pdf

%load_ext autoreload
%autoreload 2

import Components as comp
import socket
import sys

if socket.gethostname() == "ph-photonbec5":
    sys.path.append(r"D:/Control/PythonPackages/")

components = dict()

#filter_wheel = comp.FilterWheel(allowed_filter_positions=[0, 1, 2, 3, 4, 5])
#laser = comp.Toptica_Laser()

def save_arrays(channel_data, time_array, stage_position, laser_power, base_directory='D:/Data'):
    """
    Saves voltage data from multiple channels and the corresponding time array.
    
    Parameters:
        channel_data (dict): Dictionary with channel names as keys and voltage data as values.
        time_array (np.ndarray): Time axis array.
        stage_position (str): Stage position for file naming.
        laser_power (str): Laser power for file naming.
        base_directory (str): Base directory for saving data."""
    # Get current year, year+month, and year+month+day in the desired format
    year = datetime.now().strftime('%Y')
    year_month = datetime.now().strftime('%Y%m')
    year_month_day = datetime.now().strftime('%Y%m%d')
    # Define the folder path following the structure Year/YearMonth/YearMonthDay
    folder_path = os.path.join(base_directory, year, year_month, year_month_day)
    # Create the directory structure if it doesn't exist
    os.makedirs(folder_path, exist_ok=True)
    # Create timestamp for file naming
    timestamp = datetime.now().strftime('%H%M%S')  # Using HHMMSS format for timestamp
    # Save each channel's data with a distinctive file name
    for channel, voltage_array in channel_data.items():
        # Generate filename for the channel data
        filename_voltage = f"pbec_{timestamp}_{stage_position}_{laser_power}_channel{channel}.txt"
        # Define full path
        array_path_voltage = os.path.join(folder_path, filename_voltage)
        # Save voltage data for this channel
        np.savetxt(array_path_voltage, voltage_array, fmt='%0.4f')
    # Save the time array with a distinctive name
    filename_time = f"pbec_{timestamp}_{stage_position}_{laser_power}_Time.txt"
    time_path = os.path.join(folder_path, filename_time)
    np.savetxt(time_path, time_array, fmt='%0.4f')

def save_arrays2(channel_data, time_array, stage_position, laser_power, i, base_directory='D:/Data'):
    """
    Saves voltage data from multiple channels and the corresponding time array.

    Parameters:
        channel_data (dict): Dictionary with channel names as keys and voltage data as values.
        time_array (np.ndarray): Time axis array.
        stage_position (str): Stage position for file naming.
        laser_power (str): Laser power for file naming.
        base_directory (str): Base directory for saving data."""
    # Get current year, year+month, and year+month+day in the desired format
    year = datetime.now().strftime('%Y')
    year_month = datetime.now().strftime('%Y%m')
    year_month_day = datetime.now().strftime('%Y%m%d')
    # Define the folder path following the structure Year/YearMonth/YearMonthDay
    folder_path = os.path.join(base_directory, year, year_month, year_month_day)
    # Create the directory structure if it doesn't exist
    os.makedirs(folder_path, exist_ok=True)
    # Create timestamp for file naming
    timestamp = datetime.now().strftime('%H%M%S')  # Using HHMMSS format for timestamp
    # Save each channel's data with a distinctive file name
    for channel, voltage_array in channel_data.items():
        # Generate filename for the channel data
        filename_voltage = f"pbec_{timestamp}_{stage_position}_{laser_power}_channel{channel}_iteration{i}.txt"
        # Define full path
        array_path_voltage = os.path.join(folder_path, filename_voltage)
        # Save voltage data for this channel
        np.savetxt(array_path_voltage, voltage_array, fmt='%0.4f')
    # Save the time array with a distinctive name
    filename_time = f"pbec_{timestamp}_{stage_position}_{laser_power}_iteration{i}_Time.txt"
    time_path = os.path.join(folder_path, filename_time)
    np.savetxt(time_path, time_array, fmt='%0.4f')

def get_picoscope_data(
        channels=[0],  # List of channels to acquire from, e.g., [0, 1] for channels A and B
        analogue_offset=0,
        trig=0,
        trigsource=0,
        trigthresh=64,
        trigdir=2,
        trigdelay=0,
        trigms=1000,
        preTriggerSamples=2500,
        postTriggerSamples=2500,
        timebase=8,
        live_plot=False
):
    # Initialize PicoScope
    status = {}
    chandle = c_int16()
    serial = c_char_p()  # Leave serial empty for single device
    status["openunit"] = ps.ps6000OpenUnit(byref(chandle), serial)
    assert status["openunit"] == 0, f"Failed to open PicoScope 6407, error code: {status['openunit']}"
    # Channel configuration mapping
    channel_mapping = {
        0: 'PS6000_CHANNEL_A',
        1: 'PS6000_CHANNEL_B',
        2: 'PS6000_CHANNEL_C',
        3: 'PS6000_CHANNEL_D'
    }
    # Enable and configure each specified channel
    buffers = {}
    chARange = ps.PS6000_RANGE['PS6000_100MV']  # ±100 mV range, the only range available for PicoScope 6407
    for ch in channels:
        if ch not in channel_mapping:
            raise ValueError("Invalid channel number. Must be 0, 1, 2, or 3.")
        channel = ps.PS6000_CHANNEL[channel_mapping[ch]]
        enabled = 1  # 1 for enabling
        coupling_type = ps.PS6000_COUPLING['PS6000_DC_50R']  # Only DC 50Ω coupling for PicoScope 6407
        bandwidth = ps.PS6000_BANDWIDTH_LIMITER['PS6000_BW_FULL']  # Full bandwidth
        # Set channel
        status[f"set_channel_{ch}"] = ps.ps6000SetChannel(
            chandle, channel, enabled, coupling_type, chARange, analogue_offset, bandwidth
        )
        assert status[f"set_channel_{ch}"] == 0, f"Error in setting channel {ch}: {status[f'set_channel_{ch}']}"
        # Create buffers for data collection on each channel
        buffers[ch] = {
            'max': (c_int16 * (preTriggerSamples + postTriggerSamples))(),
            'min': (c_int16 * (preTriggerSamples + postTriggerSamples))()  # Used for downsampling
        }
        # Set data buffer location for data collection
        status[f"setDataBuffers_{ch}"] = ps.ps6000SetDataBuffers(
            chandle, channel, byref(buffers[ch]['max']), byref(buffers[ch]['min']), preTriggerSamples + postTriggerSamples, 0
        )
        assert_pico_ok(status[f"setDataBuffers_{ch}"])
    # Trigger setup
    status["trigger"] = ps.ps6000SetSimpleTrigger(
        chandle, trig, trigsource, trigthresh, trigdir, trigdelay, trigms
    )
    assert_pico_ok(status["trigger"])
    # Set number of pre and post trigger samples
    maxSamples = preTriggerSamples + postTriggerSamples
    # Get timebase information
    timeIntervalns = c_float()
    returnedMaxSamples = c_int32()
    status["getTimebase2"] = ps.ps6000GetTimebase2(
        chandle, timebase, maxSamples, byref(timeIntervalns), 1, byref(returnedMaxSamples), 0
    )
    assert_pico_ok(status["getTimebase2"])
    # Run block capture
    status["runBlock"] = ps.ps6000RunBlock(
        chandle, preTriggerSamples, postTriggerSamples, timebase, 0, None, 0, None, None
    )
    assert_pico_ok(status["runBlock"])
    # Wait for data collection to complete
    ready = c_int16(0)
    while ready.value == 0:
        status["isReady"] = ps.ps6000IsReady(chandle, byref(ready))
    # Retrieve data from scope to buffers for each channel
    overflow = c_int16()
    cmaxSamples = c_int32(maxSamples)
    status["getValues"] = ps.ps6000GetValues(
        chandle, 0, byref(cmaxSamples), 1, 0, 0, byref(overflow)
    )
    assert_pico_ok(status["getValues"])
    # Find maximum ADC count value
    maxADC = c_int16(32512)
    # Convert data from each channel buffer to mV and store in a dictionary
    channel_data = {}
    for ch in channels:
        adc2mV_buffer = adc2mV(buffers[ch]['max'], chARange, maxADC)
        channel_data[ch] = adc2mV_buffer
    time = np.linspace(0, (cmaxSamples.value - 1) * timeIntervalns.value, cmaxSamples.value) # Create time data
    # Plot data from each channel if live_plot is enabled
    if live_plot:
        plt.figure(figsize=(5,5))
        for ch in channels:
            plt.plot(time, channel_data[ch][:], label=f"Channel {channel_mapping[ch]}")
        plt.xlabel('Time (ns)')
        plt.ylabel('Voltage (mV)')
        plt.title("PicoScope Data Capture on Multiple Channels")
        plt.legend()
        plt.grid()
        plt.show()
    # Stop the device
    status["stop"] = ps.ps6000Stop(chandle)
    assert_pico_ok(status["stop"])
    ps.ps6000CloseUnit(chandle) # Close unit (disconnect the scope)
    return channel_data, time # Return channel data and time for further processing


THIS CODE IS FOR MULTICHANNEL PICOSCOPE COHERENCE ACQUISITION (MAKE SURE TO CHANGE PARAMETERS AND FILTER POSITIONS BEFORE SCANNING)

In [ ]:
%load_ext autoreload
%autoreload 2
import heapq
import time

if socket.gethostname() == "ph-photonbec5":
    sys.path.append("D:/Control/PythonPackages/")
    sys.path.append("D:/Control/PiezoController/")

#_________________________________________________________PARAMETERS : CHANGE HERE____________________________________________________________
#position_list = np.append(np.append(np.arange(0,40,5),np.arange(40,150,10)),np.arange(150,630,30))
#position_list = np.append(np.arange(0, 50, 5), np.arange(50, 625, 25))
position_list = np.arange(0, 250, 10)
#position_list = np.append(np.arange(0, 250, 10), np.arange(250, 650, 50))
power_list = [8/1000]

#power_list=np.arange(7.4,8,0.1)/1000
number_of_measurements = 1  # Amount of pics to take at each position of the stage

channels = [0]  # Channel A, B
time_range = 20
preSamples = 0
postSamples = 97700
plot_flag = False  # If you want to live monitor the data during the acquisition
n_avg = 50  # Number of points over which to perform averaging for Imax and Imin
DC =0
filter_positions = {
    # Preset the filter position for each power value - Need to be done manually, do it everytime before taking measurements
    45: 0,
    57: 1,
    75: 2,
    # Add more power settings as needed
}
#_________________________________________________________________________________________________________________________________

with comp.Translation_Stage() as stage:
    for power in power_list:
        #laser.set(power)
        print('Starting scan for power : ', power * 1000)
        stage.set(position_list[0])
        visibilities = []
        time.sleep(5)
        try:

            real_positions = []
            for k in range(len(position_list)):
                stage.set(position_list[k])
                position = stage.get_position()
                real_positions.append(position)

                for k in range(number_of_measurements):
                    a = get_picoscope_data(channels, live_plot=plot_flag, timebase=time_range,
                                           preTriggerSamples=preSamples, postTriggerSamples=postSamples)
                    Voltages = a[0]
                    Timing = a[1]
                    chA = np.abs(np.array(a[0][0]))
                    #bg = 0.001
                    #chC = (chA - DC) / (chB - bg)
                    highest_vals = heapq.nlargest(n_avg, chA)
                    lowest_vals = heapq.nsmallest(n_avg, chA)
                    Imax = np.mean(highest_vals)
                    Imin = np.mean(lowest_vals)
                    visibility = (Imax - Imin) / (Imax + Imin)
                    visibilities.append(visibility)
                    save_arrays(Voltages, Timing, position, power * 1000)
                    #print('Now at Position : ', position, ' / Visibility : ', visibility)
                time.sleep(1)
            plt.plot(real_positions, visibilities, 'o')
            plt.xlabel('stage position')
            plt.ylabel('visibilities')
            plt.show()
            #print(real_positions, visibilities)
        except:
            print('An exception occured')


In [3]:
 %load_ext autoreload
%autoreload 2
import heapq
import time

%load_ext autoreload
%autoreload 2

import Components as comp
import socket
import sys

if socket.gethostname() == "ph-photonbec5":
    sys.path.append(r"D:/Control/PythonPackages/")

components = dict()

filter_wheel = comp.FilterWheel(allowed_filter_positions=[0, 1, 2, 3, 4, 5])
laser = comp.Toptica_Laser()

if socket.gethostname() == "ph-photonbec5":
    sys.path.append("D:/Control/PythonPackages/")
    sys.path.append("D:/Control/PiezoController/")

#_________________________________________________________PARAMETERS : CHANGE HERE____________________________________________________________
#position_list = np.append(np.append(np.arange(0,40,5),np.arange(40,150,10)),np.arange(150,630,30))
#position_list = np.append(np.arange(0, 50, 5), np.arange(50, 625, 25))
# position_list = np.arange(0, 250, 10)
position_list = np.append(np.arange(0, 50, 10), np.arange(50, 400, 25))
#power_list = np.append(np.append(np.arange(10.7,11,0.05)/1000,np.arange(11,13,0.5)/1000),np.arange(13,25,2)/1000)
power_list = np.append(np.append(np.arange(19.7,20.2,0.05)/1000,np.arange(20.2,21.2,0.02)/1000),np.arange(21.2,23,0.3)/1000)

number_of_measurements = 5  # Amount of pics to take at each position of the stage

channels = [0,1]  # Channel A, B
time_range = 13
preSamples = 0
postSamples = 200000
plot_flag = False  # If you want to live monitor the data during the acquisition
n_avg = 50  # Number of points over which to perform averaging for Imax and Imin
DC =6.6
filter_positions = [
    (21.7, 22.60, 1),
    (22.6, 25.650, 2),
    (25.650, 26.250, 3),
    (27.35, 27.75, 4),
    (27.75, 28.50, 5),
    (28.5, float('inf'), 6)
]
#_________________________________________________________________________________________________________________________________



with comp.Translation_Stage() as stage:
    for power in power_list:
        laser.set(power)
        print('Starting scan for power : ', power * 1000)
        stage.set(position_list[0])
        visibilities = []
        time.sleep(5)
        try:
            filter_position = None
            for lower, upper, fposition in filter_positions:
                if lower <= power * 1000 < upper:
                    filter_position = fposition
                    break

            if filter_position is not None:
                filter_wheel.set(filter_position-1)
                print(f'Filter position set to: {filter_position-1} for power {power*1000}')

            real_positions = []
            for k in range(len(position_list)):
                stage.set(position_list[k])
                position = stage.get_position()
                real_positions.append(position)
                # print("position: ", position)

                for k in range(number_of_measurements):
                    a = get_picoscope_data(channels, live_plot=plot_flag, timebase=time_range,
                                           preTriggerSamples=preSamples, postTriggerSamples=postSamples)
                    Voltages = a[0]
                    Timing = a[1]
                    chA = np.abs(np.array(a[0][0]))
                    bg = 4.3
                    chB = np.abs(np.array(a[0][1]))
                    chC = (chA - DC) / (chB - bg)
                    highest_vals = heapq.nlargest(n_avg, chC)
                    lowest_vals = heapq.nsmallest(n_avg, chC)
                    Imax = np.mean(highest_vals)
                    Imin = np.mean(lowest_vals)
                    visibility = (Imax - Imin) / (Imax + Imin)
                    visibilities.append(visibility)
                    save_arrays(Voltages, Timing, position, power * 1000)
                    #print('Now at Position : ', position, ' / Visibility : ', visibility)
                time.sleep(1)
            plt.plot(real_positions, visibilities, 'o')
            plt.xlabel('stage position')
            plt.ylabel('visibilities')
            plt.show()
            #print(real_positions, visibilities)

        except:
             print('An exception occured')
    laser.set(0)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


SerialException: could not open port 'COM6': PermissionError(13, 'Access is denied.', None, 5)